# Guess the Number — Interactive Python Game

A beginner-friendly number-guessing game built in **Google Colab** using Python and `ipywidgets`.

### Features
- Three difficulty levels: Easy (1–10; 5 attempts), Medium (1–50; 7 attempts), Hard (1–100; 10 attempts)
- Buttons, a score counter, and a live remaining-lives bar
- Guess history and a record of wins/losses for the current session
- Best scores by difficulty, with **optional** Google Drive persistence
- Animated victory message and a short generated victory sound

### How to run
1. Open this notebook in [Google Colab](https://colab.research.google.com/).
2. Run the next cell to load the game and its settings.
3. Run the final cell to launch the graphical game.
4. Select a difficulty, enter a guess, and click **GUESS**. Use **PLAY AGAIN** for a new round.

> The default setting does **not** request access to your Google Drive. To reuse your saved high scores across sessions, set `SAVE_SCORES_TO_DRIVE = True` in the next cell and approve the standard Google Drive authorization when prompted. Existing scores from the earlier notebook use the same `high_scores.json` location.


In [ ]:
# STEP 1 — Imports and settings
import json
import random
from pathlib import Path

import numpy as np
import ipywidgets as widgets
from IPython.display import Audio, clear_output, display

# Change to True if you want to load/save high scores in YOUR Google Drive.
# Leave False for a GitHub demo that runs without requesting Drive access.
SAVE_SCORES_TO_DRIVE = False

SCORES_FILE = Path(
    "/content/drive/MyDrive/Guess_the_Number_Game/high_scores.json"
)

LEVELS = {
    "Easy": {"maximum": 10, "attempts": 5},
    "Medium": {"maximum": 50, "attempts": 7},
    "Hard": {"maximum": 100, "attempts": 10},
}


def score_for_correct_guess(attempt_number, max_attempts):
    """Award more points for winning with fewer guesses."""
    penalty = 100 // max_attempts
    return max(0, 100 - (attempt_number - 1) * penalty)


# Sanity checks for the scoring system.
assert score_for_correct_guess(1, 5) == 100
assert score_for_correct_guess(3, 5) == 60
assert score_for_correct_guess(5, 5) == 20


In [ ]:
# STEP 2 — Complete game (interface, score, history, sound, and storage)
class GuessTheNumberGame:
    def __init__(self, save_to_drive=False):
        self.save_to_drive = save_to_drive
        self.high_scores = {level: 0 for level in LEVELS}
        self.previous_games = []
        self.guesses = []
        self.finished = False

        if self.save_to_drive:
            self.load_high_scores()

        # Interface controls
        self.difficulty = widgets.Dropdown(
            options=list(LEVELS), description="Difficulty:"
        )
        self.guess_input = widgets.BoundedIntText(
            value=1, min=1, max=10, description="Your guess:"
        )
        self.guess_button = widgets.Button(
            description="GUESS", button_style="success", icon="check"
        )
        self.restart_button = widgets.Button(
            description="PLAY AGAIN", button_style="info", icon="refresh"
        )
        self.message = widgets.HTML()
        self.stats = widgets.HTML()
        self.guess_history = widgets.HTML()
        self.high_scores_display = widgets.HTML()
        self.previous_games_display = widgets.HTML()
        self.animation = widgets.HTML()
        self.sound = widgets.Output()
        self.lives = widgets.IntProgress(
            value=5, min=0, max=5, description="Lives left:",
            bar_style="info",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="420px")
        )

        heading = widgets.HTML('''
            <div style="background:#172554;padding:18px;border-radius:12px;
                        text-align:center;color:white">
                <h1 style="margin:0;color:white">GUESS THE NUMBER</h1>
                <p style="color:#bfdbfe;margin-bottom:0">Test your guessing skills!</p>
            </div>
        ''')

        self.ui = widgets.VBox(
            [heading, self.difficulty, self.message, self.guess_input,
             widgets.HBox([self.guess_button, self.restart_button]),
             self.stats, self.lives, self.guess_history,
             self.high_scores_display, self.previous_games_display,
             self.animation, self.sound],
            layout=widgets.Layout(
                width="540px", padding="16px", border="1px solid #cbd5e1"
            )
        )

        # Attach each action ONCE. Avoid duplicate callbacks after multiple cells.
        self.guess_button.on_click(self.check_guess)
        self.restart_button.on_click(self.new_round)
        self.difficulty.observe(self.on_difficulty_change, names="value")
        self.new_round()

    def load_high_scores(self):
        """Optional: load existing high scores from Google Drive."""
        from google.colab import drive
        drive.mount("/content/drive")
        SCORES_FILE.parent.mkdir(parents=True, exist_ok=True)
        if SCORES_FILE.exists():
            try:
                with SCORES_FILE.open("r", encoding="utf-8") as f:
                    saved = json.load(f)
                for level in LEVELS:
                    self.high_scores[level] = max(
                        0, int(saved.get(level, 0))
                    )
                print("Previous high scores loaded from Google Drive.")
            except (ValueError, TypeError, json.JSONDecodeError, OSError) as error:
                print("Could not read saved scores:", error)
        else:
            print("No saved scores found yet. Starting fresh.")

    def save_high_scores(self):
        """Only save if Drive persistence is enabled."""
        if not self.save_to_drive:
            return
        try:
            SCORES_FILE.parent.mkdir(parents=True, exist_ok=True)
            with SCORES_FILE.open("w", encoding="utf-8") as f:
                json.dump(self.high_scores, f, indent=2)
        except OSError as error:
            print("Could not save high scores:", error)

    def render_records(self):
        latest = ", ".join(str(n) for n in self.guesses) or "None yet"
        self.guess_history.value = f"<h3>Guess History</h3><p>{latest}</p>"

        entries = "".join(
            f"<li>{level}: <b>{self.high_scores[level]}</b></li>"
            for level in LEVELS
        )
        self.high_scores_display.value = (
            f"<h3>High Scores</h3><ul>{entries}</ul>"
        )

        if self.previous_games:
            entries = "".join(
                f"<li>{level}: {result} — {points} points</li>"
                for level, result, points in reversed(self.previous_games[-10:])
            )
            self.previous_games_display.value = (
                f"<h3>Previous Games (current session)</h3><ul>{entries}</ul>"
            )
        else:
            self.previous_games_display.value = (
                "<h3>Previous Games (current session)</h3><p>None yet</p>"
            )

    def new_round(self, button=None):
        self.level = self.difficulty.value
        config = LEVELS[self.level]
        self.maximum = config["maximum"]
        self.max_attempts = config["attempts"]
        self.secret = random.randint(1, self.maximum)
        self.guesses = []
        self.finished = False

        self.guess_input.max = self.maximum
        self.guess_input.value = 1
        self.guess_button.disabled = False
        self.message.value = (
            f"<h3>Guess a number from 1 to {self.maximum}!</h3>"
        )
        self.stats.value = "<b>Potential score: 100 | Attempts used: 0</b>"
        self.lives.max = self.max_attempts
        self.lives.value = self.max_attempts
        self.lives.bar_style = "info"
        self.animation.value = ""
        with self.sound:
            clear_output(wait=True)
        self.render_records()

    def on_difficulty_change(self, change):
        if change["name"] == "value":
            self.new_round()

    def play_victory_sound(self):
        # Original short victory tone generated with NumPy (no external music).
        sample_rate = 22050
        duration = 0.35
        t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
        wave = np.sin(2 * np.pi * 740 * t) * np.exp(-7 * t)
        with self.sound:
            clear_output(wait=True)
            display(Audio(wave, rate=sample_rate, autoplay=True))

    def check_guess(self, button):
        if self.finished:
            return

        guess = self.guess_input.value
        self.guesses.append(guess)
        used = len(self.guesses)
        remaining = self.max_attempts - used
        self.lives.value = remaining
        self.lives.bar_style = (
            "danger" if remaining == 0 else
            "warning" if remaining <= 2 else "info"
        )

        if guess == self.secret:
            self.finished = True
            self.guess_button.disabled = True
            points = score_for_correct_guess(used, self.max_attempts)
            self.high_scores[self.level] = max(
                self.high_scores[self.level], points
            )
            self.previous_games.append((self.level, "WON", points))
            self.save_high_scores()
            self.message.value = (
                "<h2 style='color:green'>YOU WON!</h2>"
            )
            self.animation.value = '''
                <div style="background:#dcfce7;border-radius:12px;
                            padding:18px;text-align:center;animation:pop .6s ease">
                  <style>@keyframes pop {
                    0% {transform:scale(.85);opacity:.3}
                    100% {transform:scale(1);opacity:1}
                  }</style>
                  <h2 style="color:#166534">YOU ARE A WINNER!</h2>
                  <p style="color:#166534">CONGRATULATIONS!</p>
                </div>
            '''
            self.play_victory_sound()
            self.stats.value = (
                f"<b>Final score: {points} | Attempts used: {used}</b>"
            )
        elif remaining == 0:
            self.finished = True
            self.guess_button.disabled = True
            self.previous_games.append((self.level, "LOST", 0))
            self.message.value = (
                f"<h2 style='color:#dc2626'>GAME OVER!</h2>"
                f"<p>The correct number was {self.secret}.</p>"
            )
            self.stats.value = (
                f"<b>Final score: 0 | Attempts used: {used}</b>"
            )
        else:
            hint = "Too HIGH! Try smaller." if guess > self.secret else (
                "Too LOW! Try bigger."
            )
            self.message.value = f"<h3>{hint}</h3>"
            possible = score_for_correct_guess(used + 1, self.max_attempts)
            self.stats.value = (
                f"<b>Potential score: {possible} | Attempts used: {used}</b>"
            )

        self.render_records()


In [ ]:
# STEP 3 — Run this cell to launch the game
app = GuessTheNumberGame(save_to_drive=SAVE_SCORES_TO_DRIVE)
display(app.ui)


### Notes for GitHub

- All game code is contained in this notebook. Colab includes `ipywidgets` and NumPy in a typical Python environment.
- If you run it outside Colab, install the dependencies and use a Jupyter notebook that supports `ipywidgets`.
- High scores are **session-only** by default; set `SAVE_SCORES_TO_DRIVE = True` to store them in your own Google Drive.
- *Guess History* and *Previous Games* stay in the current session only.
- This notebook contains no account credentials and does not include anyone's saved high-score file.
- Sound autoplay can be blocked by browser settings; use the audio player's play control if necessary.
